storing tables into csv 

In [42]:
import pdfplumber
import pandas as pd
from pathlib import Path

In [41]:
pdf_path = "D:/sudhendra/learning projects/GraphRAG/data/Apple23 report.pdf"

In [43]:
def make_table_row_text(row):
    clean_row = [
        str(cell).strip()
        for cell in row
        if cell and str(cell).strip()
    ]

    if len(clean_row) < 2:
        return None

    row_text = " | ".join(clean_row)

    return f"""
Financial table row.
Columns: 2023 | 2022 | 2021
Row: {row_text}
""".strip()

In [44]:
table_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        if not tables:
            continue

        for table_index, table in enumerate(tables):
            if not table:
                continue

            for row_index, row in enumerate(table):
                row_text = make_table_row_text(row)

                if row_text:
                    table_rows.append({
                        "page": page_index + 1,
                        "table_id": f"page_{page_index+1}_table_{table_index}_row_{row_index}",
                        "text": row_text
                    })

tables_df = pd.DataFrame(table_rows)

tables_df.head()

,page,table_id,text
0,3,page_3_table_0_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
1,3,page_3_table_2_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
2,3,page_3_table_3_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
3,3,page_3_table_4_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
4,3,page_3_table_5_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...


In [45]:
output_path = Path("D:/sudhendra/learning projects/GraphRAG/data/apple_2023_table_rows.csv")
tables_df.to_csv(output_path, index=False)
output_path

WindowsPath('D:/sudhendra/learning projects/GraphRAG/data/apple_2023_table_rows.csv')

In [46]:
tables_df[
    tables_df["text"].str.contains("total net sales", case=False, na=False)
]


,page,table_id,text


In [47]:
tables_df[
    tables_df["text"].str.contains("net income", case=False, na=False)
]

,page,table_id,text


In [48]:
tables_df[
    tables_df["text"].str.contains("research and development", case=False, na=False)
]

,page,table_id,text


storing data into JSON 

In [49]:
import pandas as pd
import json
import re
from pathlib import Path

In [50]:
tables_df = pd.read_csv("../data/apple_2023_table_rows.csv")
tables_df.head()

,page,table_id,text
0,3,page_3_table_0_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
1,3,page_3_table_2_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
2,3,page_3_table_3_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
3,3,page_3_table_4_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
4,3,page_3_table_5_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...


In [51]:
def clean_nums(value):
    if value is None:
        return None
    value = str(value).strip()
    negative = False

    if "(" in value and ")" in value:
        negative = True


    value = value.replace("$","")
    value = value.replace(",", "")
    value = value.replace("(", "")
    value = value.replace(")", "")
    value = value.replace("%", "")
    value = value.strip()

    try:
        number = float(value)
        return -number if negative else number

    except:
        return None



In [52]:
def extract_metric_from_row(row_text):
    row_text = str(row_text)

    if "Row:" not in row_text:
        return None

    row_part = row_text.split("Row:")[-1].strip()
    parts = [p.strip() for p in row_part.split("|") if p.strip()]

    if len(parts) < 4:
        return None

    metric_name = parts[0].lower().strip()

    values = {
        "2023": clean_nums(parts[1]),
        "2022": clean_nums(parts[2]),
        "2021": clean_nums(parts[3])
    }

    return metric_name,values


In [53]:
target_metrics = {
    "total net sales": "total_net_sales",
    "gross margin": "gross_margin",
    "operating income": "operating_income",
    "net income": "net_income",
    "research and development": "research_and_development",
    "selling, general and administrative": "selling_general_and_administrative",
    "total assets": "total_assets",
    "total liabilities": "total_liabilities",
    "cash and cash equivalents": "cash_and_cash_equivalents"
}

In [54]:
financial_metrics = {}

for _,row in tables_df.iterrows():
    text = row["text"]
    extracted = extract_metric_from_row(text)

    if extracted is None:
        continue

    metric_name,values = extracted

    for target_text,clean_key in target_metrics.items():
        if target_text in metric_name:
            financial_metrics[clean_key] = {
                "label": target_text,
                "values": values,
                "source_page": int(row["page"]),
                "source_type": "table_row"
            }

financial_metrics

{}

In [55]:
output_path = Path("../data/apple_2023_financial_metrics.json")

with open(output_path,"w") as f:
    json.dump(financial_metrics,f,indent=4)

output_path

WindowsPath('../data/apple_2023_financial_metrics.json')

In [56]:
with open("../data/apple_2023_financial_metrics.json","r") as f:
    metrics = json.load(f)

metrics

{}

In [ ]:
def get_metric(metrics,metric_name,previous,year):
    return metrics[metric_name]["values"][str(year)]

def percentage_change(current,previous):
    return ((current - previous) / previous) * 100

def margin(value,revenue):
    return (value/revenue) * 100

In [58]:
revenue_2023 = get_metric(metrics, "total_net_sales", 2023)
revenue_2022 = get_metric(metrics, "total_net_sales", 2022)

revenue_change = revenue_2023 - revenue_2022



KeyError: 'total_net_sales'

testing becuase json is not storing anything


In [40]:
for _, row in tables_df.iterrows():
    text = row["text"]

    extracted = extract_metric_from_row(text)

    if extracted is not None:
        metric_name, values = extracted

        print(metric_name)

In [59]:
print(tables_df.shape)
print(tables_df.columns)
print(tables_df.head())

(21, 3)
Index(['page', 'table_id', 'text'], dtype='str')
   page              table_id  \
0     3  page_3_table_0_row_0   
1     3  page_3_table_2_row_0   
2     3  page_3_table_3_row_0   
3     3  page_3_table_4_row_0   
4     3  page_3_table_5_row_0   

                                                text  
0  Financial table row.\nColumns: 2023 | 2022 | 2...  
1  Financial table row.\nColumns: 2023 | 2022 | 2...  
2  Financial table row.\nColumns: 2023 | 2022 | 2...  
3  Financial table row.\nColumns: 2023 | 2022 | 2...  
4  Financial table row.\nColumns: 2023 | 2022 | 2...  


In [60]:
for i in range(5):
    print("ROW", i)
    print(tables_df.loc[i, "text"])
    print("-" * 100)

ROW 0
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 1. | Business | 1
----------------------------------------------------------------------------------------------------
ROW 1
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 10. | Directors, Executive Officers and Corporate Governance | 53
----------------------------------------------------------------------------------------------------
ROW 2
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 12. | Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters | 53
----------------------------------------------------------------------------------------------------
ROW 3
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 14. | Principal Accountant Fees and Services | 53
----------------------------------------------------------------------------------------------------
ROW 4
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 15. | Exhibit and Financial Sta

In [61]:
import re

def clean_number(value):
    if value is None:
        return None

    value = str(value).strip()

    negative = False
    if "(" in value and ")" in value:
        negative = True

    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace("(", "")
    value = value.replace(")", "")
    value = value.replace("%", "")
    value = value.strip()

    try:
        number = float(value)
        return -number if negative else number
    except:
        return None


def extract_metric_from_row(row_text):
    row_text = str(row_text).strip()

    # If your text contains "Row:", take only the part after Row:
    if "Row:" in row_text:
        row_text = row_text.split("Row:")[-1].strip()

    # Extract numbers like 383,285 or $ 383,285 or (11,043)
    number_strings = re.findall(r"\(?\$?\s?\d[\d,]*\.?\d*\)?", row_text)

    numbers = []
    for n in number_strings:
        cleaned = clean_number(n)
        if cleaned is not None:
            numbers.append(cleaned)

    # Need at least 3 year values: 2023, 2022, 2021
    if len(numbers) < 3:
        return None

    # Remove numbers from row to get metric name
    metric_name = re.sub(r"\(?\$?\s?\d[\d,]*\.?\d*\)?", "", row_text)
    metric_name = metric_name.replace("|", " ")
    metric_name = re.sub(r"\s+", " ", metric_name).strip().lower()

    values = {
        "2023": numbers[0],
        "2022": numbers[1],
        "2021": numbers[2]
    }

    return metric_name, values

In [63]:
matches = tables_df[
    tables_df["text"].str.contains("total net sales", case=False, na=False)
]

for _, row in matches.iterrows():
    print(row["text"])
    print(extract_metric_from_row(row["text"]))
    print("-" * 100)

another testing

In [64]:
print(tables_df.shape)
print(tables_df.columns)
tables_df.head(10)

(21, 3)
Index(['page', 'table_id', 'text'], dtype='str')


,page,table_id,text
0,3,page_3_table_0_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
1,3,page_3_table_2_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
2,3,page_3_table_3_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
3,3,page_3_table_4_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
4,3,page_3_table_5_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
5,26,page_26_table_1_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
6,30,page_30_table_0_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...
7,30,page_30_table_0_row_2,Financial table row.\nColumns: 2023 | 2022 | 2...
8,30,page_30_table_0_row_4,Financial table row.\nColumns: 2023 | 2022 | 2...
9,41,page_41_table_0_row_0,Financial table row.\nColumns: 2023 | 2022 | 2...


In [65]:
for i in range(20):
    print("ROW", i)
    print(tables_df.loc[i, "text"])
    print("-" * 100)

ROW 0
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 1. | Business | 1
----------------------------------------------------------------------------------------------------
ROW 1
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 10. | Directors, Executive Officers and Corporate Governance | 53
----------------------------------------------------------------------------------------------------
ROW 2
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 12. | Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters | 53
----------------------------------------------------------------------------------------------------
ROW 3
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 14. | Principal Accountant Fees and Services | 53
----------------------------------------------------------------------------------------------------
ROW 4
Financial table row.
Columns: 2023 | 2022 | 2021
Row: Item 15. | Exhibit and Financial Sta

In [66]:
tables_df[
    tables_df["text"].str.contains("sales", case=False, na=False)
][["page", "text"]].head(20)

,page,text


In [67]:
print(tables_df.shape)

(21, 3)


aborting this while file and going back to table extraction in new ipynb file